# **❓ Missing Values**

<img src="assets/banner_missing_values.png" style="width:95%">

| **Name**             | **Type**     | **Description**                                                                                   |
| -------------------- | ------------ | ------------------------------------------------------------------------------------------------- |
| **`IterativeImputer`** | Multivariate | Estimates each feature with missing values as a function of other features iteratively.           |
| **`KNNImputer`**       | Multivariate | Fills missing values using values from the nearest neighbors (based on feature similarity).       |
| **`MissingIndicator`** | Transformer  | Adds binary columns indicating the locations of missing values (0 = present, 1 = missing).        |
| **`SimpleImputer`**    | Univariate   | Replaces missing values using simple strategies such as mean, median, most frequent, or constant. |

└─ `Scikit-learn Impute API`: https://scikit-learn.org/stable/api/sklearn.impute.html

In [19]:
import numpy as np
import pandas as pd

---
---
# **🚀 Inititialize Missing Values**

- Introduce `NaN` values randomly or in a controlled manner to simulate missing data scenarios.

- Enable experimentation with different imputation strategies (mean, median, mode, KNN, iterative) on a realistic dataset.

- Facilitate understanding of the effects of missing values on model training, feature distributions, and downstream performance.

---
## **└─ Manually create values**

- Create a toy DataFrame containing true missing values (`pd.NA`, `None`) alongside placeholder values (`999`, `unknown`) for controlled testing.

- Provide a foundation to practice replacement and cleaning strategies, ensuring robust handling of heterogeneous missing-data representations.

In [ ]:
df_missing_manual = pd.DataFrame({
    'age': [25, 30, pd.NA, 45, 999],
    'income': [50000, 60000, 70000, None, 55000],
    'gender': ['male', 'female', 'unknown', 'female', 'male']
})

df_missing_manual

---
## **└─ Randomly mask a %**

- `np.random.rand`: https://numpy.org/doc/2.1/reference/random/generated/numpy.random.rand.html

- `df.mask`: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mask.html

In [ ]:
%config InteractiveShell.ast_node_interactivity = "all"

import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

data = load_iris(as_frame=True)
df = data.data

missing_frac = 0.1  # 10% missing
missing_value = pd.NA
np.random.seed(42)

mask = np.random.rand(*df.shape) < missing_frac
df_missing_random = df.mask(mask, missing_value)

df_missing_random.isnull().sum().sum()
df_missing_random

---
---
# **🔎 Identifying Missing Values**

- Detect missing values in each column using methods like .isnull() or .isna().

- Quantify the extent of missing data with counts or percentages to prioritize handling.

- Gain insights into patterns of missingness (random vs. systematic) to guide imputation or removal strategies.

| Value    | Used for                       | Notes                        |
| -------- | ------------------------------ | ---------------------------- |
| `pd.NA`  | General-purpose missing value  | Preferred for new code       |
| `np.nan` | Floats, legacy missing marker  | Only works well with float   |
| `None`   | Object dtype (strings, Python) | Converts to `pd.NA`/`np.nan` |
| `NaT`    | Datetime / Timedelta           | Temporal missing             |


---
### └─ **DataFrame Summary**

`df.info`: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html

In [ ]:
df_missing = pd.DataFrame({
    'age': [25, 30, pd.NA, 45, 999],
    'income': [50000, 60000, 70000, None, 55000],
    'gender': ['male', 'female', 'unknown', 'female', 'male']
})

df_missing.info()

---
### └─ **Count of Missing Values**

`df.isnull().sum()`: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isnull.html

In [ ]:
df_missing = pd.DataFrame({
    'age': [25, 30, pd.NA, 45, 999],
    'income': [50000, 60000, 70000, None, 55000],
    'gender': ['male', 'female', 'unknown', 'female', 'male']
})

df_missing.isnull().sum()

---
### └─ **Visual Representation**

- Using the `msno` package

In [ ]:
import missingno as msno

msno.matrix(df_missing_random)

---
### └─ **Correlation Heatmap**

- Using the `msno` package

In [ ]:
import missingno as msno

msno.heatmap(df_missing_random)

---
---
# **🛠️ Wrangling Missing Values**

- Detect missing values in each column using methods like .isnull() or .isna().

- Quantify the extent of missing data with counts or percentages to prioritize handling.

- Gain insights into patterns of missingness (random vs. systematic) to guide imputation or removal strategies.

---
### └─ **Replace Placeholders**

`df.replace`: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.replace.html

In [ ]:
df_missing = pd.DataFrame({
    'age': [25, 30, -1, 45, 999],
    'income': [50000, 60000, 70000, -999, 55000],
    'gender': ['male', 'female', 'missing', 'female', 'male']
})

df_missing_replaced = df_missing.replace({
    'age': -1,
    'income': -999,
    'gender': 'missing'
}, pd.NA)

df_missing_replaced

---
### └─ **Mark missing values**

`df.isna()`: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html#pandas.DataFrame.isna

`df.notna()`: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.notna.html#pandas.DataFrame.notna

In [ ]:
df_missing = pd.DataFrame({
    'age': [25, 30, pd.NA, 45, 999],
    'income': [50000, 60000, '70000', None, 55000],
    'gender': ['male', 'female', pd.NA, 'female', 'male']
})

df_missing.isna()

In [ ]:
df_missing = pd.DataFrame({
    'age': [25, 30, pd.NA, 45, 999],
    'income': [50000, 60000, '70000', None, 55000],
    'gender': ['male', 'female', pd.NA, 'female', 'male']
})

df_missing.notna()

---
### └─ **Create Binary Column**

- `MissingIndicator`: https://scikit-learn.org/stable/modules/generated/sklearn.impute.MissingIndicator.html

- A transformer that creates a binary mask (0 = not missing, 1 = missing) for missing values in your dataset.

- This is useful when you don’t just want to impute missing values, but also want to keep track of which values were originally missing as extra features.

In [22]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, MissingIndicator

# Example dataset with NaNs
df = pd.DataFrame({
    "feature1": [1, 3, np.nan, 7],
    "feature2": [2, np.nan, 6, 8],
    "feature3": [np.nan, 1, 2, 3]
})

print("Original DataFrame:")
print(df)

# Step 1: Initialize MissingIndicator
indicator = MissingIndicator(missing_values=np.nan)

# Step 2: Fit and transform
mask = indicator.fit_transform(df)

# Step 3: Get the columns that had missing values
missing_cols = [df.columns[i] + "_missing" for i in indicator.features_]

# Step 4: Create a DataFrame of indicators
df_indicators = pd.DataFrame(mask, columns=missing_cols, index=df.index)

# Step 5: Concatenate with original df
df_with_indicators = pd.concat([df, df_indicators], axis=1)

print("\nDataFrame with Missing Indicator Columns:")
df_with_indicators


Original DataFrame:
   feature1  feature2  feature3
0       1.0       2.0       NaN
1       3.0       NaN       1.0
2       NaN       6.0       2.0
3       7.0       8.0       3.0

DataFrame with Missing Indicator Columns:


,feature1,feature2,feature3,feature1_missing,feature2_missing,feature3_missing
0,1.0,2.0,NaN,False,False,True
1,3.0,NaN,1.0,False,True,False
2,NaN,6.0,2.0,True,False,False
3,7.0,8.0,3.0,False,False,False


---
---
# **🚮 Removal**

---
### └─ **Remove rows**

`df.dropna`: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html

In [ ]:
df_missing = pd.DataFrame({
    'age': [25, 30, pd.NA, 45, 999],
    'income': [50000, 60000, '70000', None, 55000],
    'gender': ['male', 'female', pd.NA, 'female', 'male']
})

print("Original Shape:", df_missing.shape)
df_missing_values_dropped = df_missing.dropna(axis=0)
df_missing_values_dropped

---
### └─ **Remove columns**

`df.dropna`: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html

In [ ]:
df_missing = pd.DataFrame({
    'age': [25, 30, pd.NA, 45, 999],
    'income': [50000, 60000, 70000, 75000, 55000],
    'gender': ['male', 'female', pd.NA, 'female', 'male']
})
print("Original Shape:", df_missing.shape)

df_missing_values_dropped = df_missing.dropna(axis=1)

df_missing_values_dropped

---
---
# **🔢 Univariate Imputation**

---
### └─ **Impute with Aggregate**

- `SimpleImputer`: https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html

- SimpleImputer with `strategy="mean"` doesn’t support `pd.NA` directly.

- It only works with numeric `np.nan` values.

- `pd.NA` is a nullable type introduced in recent Pandas versions and needs to be converted first.

In [6]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

df_missing = pd.DataFrame({
    'age': [25, 30, pd.NA, 45, 999],
    'income': [50000, 60000, 70000, 75000, 55000]
})

df_missing = df_missing.replace([pd.NA, 999], np.nan)

imputer = SimpleImputer(strategy="mean")
df_missing_imputed = imputer.fit_transform(df_missing)
df_missing_imputed = pd.DataFrame(df_missing_imputed, columns=df_missing.columns)
df_missing_imputed

C:\Users\joeln\AppData\Local\Temp\ipykernel_23740\2459007798.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_missing = df_missing.replace([pd.NA, 999], np.nan)


,age,income
0,25.000000,50000.0
1,30.000000,60000.0
2,33.333333,70000.0
3,45.000000,75000.0
4,33.333333,55000.0


---
### └─ **Impute with logic from other columns**

In [ ]:
import pandas as pd

def fill_missing_names(df, id_col, target_col):
    """
    Fill missing values in a target column using mappings from an ID column.

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe containing the data.
    id_col : str
        The column containing unique identifiers (e.g., town_id, flatm_id).
    target_col : str
        The column with missing values to fill (e.g., town_name, flatm_name).

    Returns
    -------
    pd.DataFrame
        A copy of the dataframe with missing values filled in the target column.
    """
    df = df.copy()

    # Build mapping from id_col → target_col using rows where target_col is not null
    mapping = df.dropna(subset=[target_col]).drop_duplicates(subset=[id_col]) \
                 .set_index(id_col)[target_col].to_dict()

    # Fill missing values using the mapping
    df[target_col] = df.apply(
        lambda row: mapping.get(row[id_col], row[target_col]),
        axis=1
    )

    return df

In [ ]:
# Example dataframe
data = {
    "town_id": [26, 26, 2, 2, 18, 18],
    "town_name": ["Yishun", None, "Bedok", None, "Punggol", None]
}
df = pd.DataFrame(data)

# Fill missing names
df_filled = fill_missing_names(df, "town_id", "town_name")
print(df_filled)

---
---
# **🌐 Multivariate Imputation**

---
### └─ **Impute with KNN**

- `KNNImputer`: https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html

- A generic imputation technique that can work on numerical data (and sometimes encoded categorical data).

- Imputes missing values based on the values of the nearest neighbors (using Euclidean or other distance metrics).

- Good for numerical data, but less straightforward or accurate for categorical data unless categories are encoded carefully.

In [9]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer

df_missing = pd.DataFrame({
    'age': [25, 30, pd.NA, 45, 999],
    'income': [50000, 60000, 70000, 75000, 55000]
})

df_missing = df_missing.replace([pd.NA, 999], np.nan)

imputer = KNNImputer(n_neighbors=2, weights='uniform')
df_missing_imputed = imputer.fit_transform(df_missing)
df_missing_imputed = pd.DataFrame(df_missing_imputed, columns=df_missing.columns)
df_missing_imputed

C:\Users\joeln\AppData\Local\Temp\ipykernel_23740\2636105310.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_missing = df_missing.replace([pd.NA, 999], np.nan)


,age,income
0,25.0,50000.0
1,30.0,60000.0
2,37.5,70000.0
3,45.0,75000.0
4,27.5,55000.0


---
### └─ **Impute with K-Modes**

- `KModes`: https://pypi.org/project/kmodes/

- Designed specifically for categorical data clustering.

- Uses a simple matching dissimilarity measure (count of mismatches) for categorical variables.

- Best when your dataset (or clustering variables) are purely categorical.

- Can be used for imputation by assigning cluster modes to missing values.

In [16]:
%config InteractiveShell.ast_node_interactivity = "all"

import numpy as np
import pandas as pd
from kmodes.kmodes import KModes

df_missing = pd.DataFrame({
    'color': ['red', 'blue', 'unknown', 'green', 'red'],
    'shape': ['circle', 'square', 'circle', 'unknown', 'triangle'],
    'size': ['S', 'M', 'L', 'M', 'unknown']
})

df_missing = df_missing.replace("unknown", np.nan)

# KModes requires strings (no NaN) → temporarily fill NaN with "missing"
df_for_clustering = df_missing.fillna('missing').astype('str')

km = KModes(n_clusters=2, init="Huang", n_init=5, random_state=42)
clusters = km.fit_predict(df_for_clustering)

print("Cluster Modes:")
cluster_modes = pd.DataFrame(km.cluster_centroids_, columns=df_missing.columns)
cluster_modes

df_missing["cluster"] = clusters
for col in df_missing.columns[:1]:
    df_missing[col] = df_missing.apply(
        lambda row: cluster_modes.loc[row["cluster"], col] if pd.isna(row[col]) else row[col],
        axis=1
    )

df_missing = df_missing.drop(columns="cluster")
print(df_missing)


Cluster Modes:


,color,shape,size
0,red,circle,L
1,blue,missing,M


   color     shape size
0    red    circle    S
1   blue    square    M
2    red    circle    L
3  green       NaN    M
4    red  triangle  NaN


---
### └─ **Impute with K-Prototypes**

- `KPrototypes`: https://pypi.org/project/kmodes/

- Extension of K-Modes that can handle mixed data types (both categorical and numerical).

- Combines Euclidean distance for numeric features + matching dissimilarity for categorical ones.

- If your dataset has both numerical and categorical features, this is often the better choice.

- Clustering with K-Prototypes better captures similarities in mixed-type datasets.

In [18]:
%config InteractiveShell.ast_node_interactivity = "all"

import numpy as np
import pandas as pd
from kmodes.kprototypes import KPrototypes

# 🔹 Example dataset with mixed data types
df_missing = pd.DataFrame({
    'color': ['red', 'blue', 'unknown', 'green', 'red'],   # categorical
    'shape': ['circle', 'square', 'circle', 'unknown', 'triangle'],  # categorical
    'size': ['S', 'M', 'L', 'M', 'unknown'],  # categorical
    'weight': [1.2, 3.4, np.nan, 2.1, 5.5]   # numeric
})

# Replace placeholders with NaN
df_missing = df_missing.replace("unknown", np.nan)

# Identify categorical vs numerical columns
categorical_cols = ['color', 'shape', 'size']
numerical_cols = ['weight']

# 🔹 Fill NaNs temporarily for clustering
df_for_clustering = df_missing.copy()
df_for_clustering[categorical_cols] = df_for_clustering[categorical_cols].fillna('missing').astype(str)
df_for_clustering[numerical_cols] = df_for_clustering[numerical_cols].fillna(-999)  # temporary placeholder

# KPrototypes requires numpy array input
data_matrix = df_for_clustering.to_numpy()

# Fit KPrototypes (must specify categorical column indices)
kproto = KPrototypes(n_clusters=2, init="Huang", n_init=5, random_state=42)
clusters = kproto.fit_predict(data_matrix, categorical=[0, 1, 2])

print("Cluster Modes & Means:")
cluster_modes = pd.DataFrame(kproto.cluster_centroids_, columns=df_for_clustering.columns)
print(cluster_modes)

# Add cluster labels
df_missing["cluster"] = clusters

# 🔹 Impute missing values using cluster centers
for col in df_missing.columns[:-1]:  # skip "cluster"
    df_missing[col] = df_missing.apply(
        lambda row: cluster_modes.loc[row["cluster"], col] if pd.isna(row[col]) else row[col],
        axis=1
    )

df_missing = df_missing.drop(columns="cluster")
print("\nImputed DataFrame:")
print(df_missing)


Cluster Modes & Means:
    color    shape    size weight
0    3.05      red  circle      M
1  -999.0  missing  circle      L

Imputed DataFrame:
    color     shape    size weight
0     red    circle       S    1.2
1    blue    square       M    3.4
2  -999.0    circle       L      L
3   green       red       M    2.1
4     red  triangle  circle    5.5


---
---
---